<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #1e3c72; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️Manipulation de données satellitaires - Format TIF</div>

### 🎯Analyser une image satellitaire TIF géoréférencée
### 🎯Reprojeter & découper
### 🎯Ajouter des couches et annotations
### 🎯Faire une animation

## Workflow : lignes de commandes bash : GDAL & ImageMagick 

Ce TP utilise en grande partie les logiciels de la librairie **"gdal"** qui signifie : *geospatial data abstraction library*. Cette librairie est extrêmement utile pour manipuler les données issues des satellites météorologiques.

Pour ce TP nous allons utiliser des fichiers au format **TIF** issus de la production opérationnelle de Météo-France.  </br>


<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Tout d'abord, il faut procéder à l'importation des librairies nécessaires à ce TP.
</div>

In [ ]:
from datetime import datetime
import sys
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

Le premier fichier TIF proposé est le suivant : EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT </br>
Il se trouve dans ce répertoire : **`/stockage/DATA/TIF/20260422`**

In [ ]:
INPUT="/stockage/DATA/TIF/20260422/EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT"
print(f"Fichier source configuré : {INPUT}")

<div class="alert alert-info alert-success">
<h3> 1 - 🔎 Visualiser le tif et son contenu  </h3>
</div>

La commande gdalinfo permet d'afficher les informations qui se trouvent dans le fichier NetCDF. Usage : `gdalinfo mon_fichier`

Pour lancer les commandes gdal dans ce jupyter notebook il faut les précéder d'un point d'exclamation : `!gdalinfo`

Maintenant, faites un gdalinfo du fichier NetCDF.

In [ ]:
!gdalinfo $INPUT

Il y a énormément d'informations proposées. Mais nous n'allons pas toutes les exploiter , en voici quelques unes

Driver: GTiff/GeoTIFF
→ Format du fichier : GeoTIFF (image raster géoréférencée standard)

Files: /stockage/DATA/TIF/20260422/EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT
→ Nom du fichier indiquant : produit Meteosat true color, date 2026-04-22 12:00 UTC

Size is 22272, 22272
→ Image carrée très hau​te résolution : 22272 × 22272 pixels (~500 m/pixel)

METHOD["Geostationary Satellite (Sweep Y)"]
→ Projection satellite géostationnaire (vue depuis l’espace)

PARAMETER["Longitude of natural origin",0
→ Satellite positionné à 0° de longitude (au-dessus de Greenwich)

PARAMETER["Satellite Height",35786400
→ Altitude du satellite : 35 786 km (orbite géostationnaire)

CS[Cartesian,2]
→ Coordonnées projetées en mètres (plan cartésien)

Origin = (-5568000.000000000000000,5568000.000000000000000)
→ Coin haut gauche de l’image (en mètres)

Pixel Size = (500.000000000000000,-500.000000000000000)
→ Résolution : 500 m par pixel (Y négatif = origine en haut)

Metadata:
→ Métadonnées du produit

TIFFTAG_IMAGEDESCRIPTION={"process":"geostationary_meteosat_0_deg","aggregate":"geostationary_true_color", "time":"2026-04-22T12:00:00Z"}
→ Produit : image Meteosat true color, centrée sur 0°, date 12:00 UTC

AREA_OR_POINT=Area
→ Chaque pixel représente une surface (et non un point)

SOURCE_COLOR_SPACE=YCbCr
→ Espace couleur YCbCr (compression JPEG)

COMPRESSION=YCbCr JPEG
→ Compression avec perte (JPEG)

JPEG_QUALITY=95
→ Qualité élevée (peu de perte visible)

Corner Coordinates:
→ Coordonnées des coins de l’image

Upper Left  (-5568000.000, 5568000.000)
→ Coin haut gauche

Lower Left  (-5568000.000,-5568000.000)
→ Coin bas gauche

Upper Right ( 5568000.000, 5568000.000)
→ Coin haut droit

Lower Right ( 5568000.000,-5568000.000)
→ Coin bas droit

Center      (   0.0000000,   0.0000000) (0°E, 0°N)
→ Centre du disque : intersection équateur / Greenwich

Band 1 Block=256x256 Type=Byte, ColorInterp=Red
→ Bande rouge (8 bits)

Band 2 Block=256x256 Type=Byte, ColorInterp=Green
→ Bande verte (8 bits)

Band 3 Block=256x256 Type=Byte, ColorInterp=Blue
→ Bande bleue (8 bits)

Overviews: 11136x11136, 5568x5568, 2784x2784, 1392x1392
→ Pyramides internes pour affichage rapide (zoom multi-échelles)

In [ ]:
cd ~/MF_DATA_MANIPULATION

Définissons la variable du dossier où seront stockés les résultats.

In [ ]:
!mkdir -p RESULTS
output = 'RESULTS'

Nous allons redimensionner l'image et la convertir en jpg à l'aide la commande suivante :

In [ ]:
!convert -quiet -resize 700x700 $INPUT[3]  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg #>/dev/null 2>&1
# noter ici que l'overview numero 3 [3] a été ulitlisée pour un gain de temps (Band 2784x2784 du tif)

## 🖼️ Il est alors possible de visualiser l'image à l'aide la commande suivante :

In [ ]:
im = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg', 'r')
display(im)

Image Geocolor RGB, en vue geos, depuis le satellite Meteosat-12 positionnée en 0° 0°. </br>
Il s'agit d'un fichier type d'une production CMS envoyée par Transmet. Réalisée à partir des fichiers multicanaux, avec divers corrections appliquées.

<div class="alert alert-info alert-success">
<h3> 2 - 🌐 Différents formats </h3>
</div>

Le format jpg est un format très utilisé car le rendu visuel est très bon et c'est un format léger</br>
Son inconvénient : il est compressé, avec une qualité 92 par défaut, et même en forçant une compression à 100 il y a quand même une altération.</br>
Pour ne pas avoir de compression, il faut faire du tif ou du png.</br>
Les différences visuelles sont souvent très ténues, voire invisibles, l'interpolation par défaut "Lanczos" est un très bon compromis netteté / aliasing (pixellisation visible).</br>
Mais les poids sont très différents :
Exemples de poids de formats :

In [ ]:
!convert -resize 700x700 $INPUT[3]  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg  >/dev/null 2>&1 
!convert -resize 700x700 $INPUT[3]  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif  >/dev/null 2>&1 
!convert -resize 700x700 $INPUT[3]  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.png  >/dev/null 2>&1 
!identify -quiet -format "%f : %b\n" $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.png

Le format png est environ 4 fois plus lourd que le jpg. Le format tif est environ 7 fois plus lourd que le jpg.

In [ ]:
!gdalinfo $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif

###  Le géoréférencement est perdu.
L'utilisation de la commande convert a fait perdre le géoréférencement. </br>
Pour le conserver il utiliser la commande gdalwarp et produire un autre TIF, qui est le seul de ces 3 formats qui garde le géoréférencement.</br>
Cette commande crée une nouvelle image en 700x700 pixels.

In [ ]:
!gdalwarp -overwrite -ts 700 700 $INPUT  $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif  

In [ ]:
!gdalinfo $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif

On retrouve les métadonnées de géoréférencement.

Il est possible de compresser un tif. </br>
La compression LZW est "sans perte", mais le gain de poids est faible. </br>
La compression YCbCr (Luminance/couleur) est très efficace. Le gain en poids est important, mais il y a des pertes (infimes, souvent invisibles, mais existantes)

In [ ]:
!gdalwarp -overwrite -ts 700 700 /stockage/DATA/TIF/20260422/EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_compress_YCbCr.tif -co COMPRESS=JPEG -co JPEG_QUALITY=95 -co PHOTOMETRIC=YCBCR >/dev/null 2>&1
!gdalwarp -overwrite -ts 700 700 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_compress_LZW.tif -co COMPRESS=LZW -co PREDICTOR=2  #>/dev/null 2>&1
!identify -quiet -format "%f : %b\n" $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_compress_YCbCr.tif 
!identify -quiet -format "%f : %b\n" $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_compress_LZW.tif 

Le poids du tif compressé LZW est ici environ 30% moins élevé que le tif non compressé. </br>
Le poids du tif compressé YCbCr est ici environ 4 fois plus petit que le tif non compressé.

👉 La compression est un paramètre très important à prendre en compte, que cela soit en ce qui concerne la perte d'information, ou le volume à manipuler.</br>
Par exemple, une production plein disque Meteosat-12, à 500m, au format tif compressé YCbCr de 160 Mo, pèse 1,4 Go non compressée.

<div class="alert alert-info alert-success">
<h3> 3 - 🌐 Reprojections </h3>
</div>

Gdal permet de faire des reprojections :

🗺️ Exemple de  projection en **EPSG:4326 WGS84** (World Geodetic System 1984)
Système de coordonnées basé sur la latitude et la longitude, unités en degrés (°). (EPSG:4326: Système géographique)</br>
Correspond visuellement à une cylindrique équidistante ou plate carrée

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:4326 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_4326.tif
!gdalwarp -overwrite -t_srs EPSG:4326 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_4326.jpg

In [ ]:
!gdalinfo $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_4326.tif

On retrouve le système  WGS 84, </br>
Les coordonnées en latitude / longitude (degrés)

In [ ]:
im2 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_4326.jpg', 'r')
display(im2)

---</br>
🧭 Autre exemple de reprojection : **Mercator** </br>
Projection cartographique de la Terre dite « cylindrique ». </br>
Référence dans le monde grâce à sa précision pour les voyages marins du fait de la conservation des angles qu'elle permet, essentielle pour prévoir les routes de navigation.

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:3857 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_3857.jpg

In [ ]:
im3 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_3857.jpg', 'r')
display(im3)

---</br>
Autre exemple de reprojection : **Peters** </br>
ou projection de Gall-Peters, est une projection cartographique qui, contrairement à la projection de Mercator, permet de prendre en compte la superficie réelle des continents

In [ ]:
!gdalwarp -overwrite -t_srs "+proj=cea +lon_0=0 +lat_ts=45 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs" $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_peter.jpg

In [ ]:
im3 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_peter.jpg', 'r')
display(im3)

<div class="alert alert-info alert-success">
<h3> 4 - Découpes </h3>
</div>

Exemple en **4326**.</br>
L'avantage de cette projection est qu'il est possible d'indiquer en entrée les coordonnées en degrés des limites de domaine (bbox)

Exemple France avec un ratio 16/9 pour coordonnées du domaine : -te {coord_ouest} {coord_sud} {coord_est} {coord_nord} </br>
L'option -ts force la taille de sortie. </br>
La source doit être un fichier tif géoréférencé. </br>
Ici un fichier déjà en EPSG:4326 WGS 84</br>
option -t_srs pour la reprojection</br>
option -te pour indiquer la zone de découpe : dans l'ordre W S E N</br>
option -ts pour indiquer la taille de sortie</br>

 ou en 1 seule ligne à partir d'un fichier tif géoréférencé quelconque


In [ ]:
!time gdalwarp -overwrite -t_srs EPSG:4326 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326.tif 
!gdalwarp -overwrite -te -14 38 18 56 -ts 960 540  $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.jpg #W S E N en projection 16/9 

Cette commande prend beaucoup de temps, car la première étape reprojette le plein disque, et ensuite réalise la découpe. </br>
💡 Gdal permet de réaliser une découpe directement reprojettée, avec un gros gain de temps.

In [ ]:
!time gdalwarp -overwrite -t_srs EPSG:4326 -te -14 38 18 56 -ts 960 540 /stockage/DATA/TIF/20260422/EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.jpg #W S E N en projection 16/9

In [ ]:
im4 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france.jpg', 'r')
display(im4)

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:4326 -te -14 38 18 56 -ts 300 540 /stockage/DATA/TIF/20260422/EOXF71MTGG221200.globeMtruecolorGTP.mtg.20260422.1200.24.tif.LT $output/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france_300x540.jpg #W S E N en projection 16/9
im5 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_4326_france_300x540.jpg', 'r')
display(im5)

### ⚠️ Attention ! L'image peut être déformée si le ratio de sortie est différent de celui de la zone de découpe

Exemple de découpe en projection **ortho** </br>
Une projection orthographique ("ortho") est une façon de représenter la Terre comme si on la regardait depuis très loin dans l'espace, à la verticale du point.</br>
Il faut ici le centre du domaine, et si nécessaire la dimension du domaine, en mètres.

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -ts 960 540 -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -ts 960 540 -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.jpg
im6 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.jpg', 'r')
display(im6)

In [ ]:
!gdalinfo $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif

Avec cette commande imposant une taille de sortie en pixel, on s'aperçoit que la résolution spatiale de sortie est de 2333x2333 m pour un pixel (Pixel Size) </br> Nous avons donc perdu en résolution spatiale


Sans taille de sortie imposée, la pleine résolution est utilisée

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_HD.tif
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_HD.jpg

In [ ]:
!gdalinfo $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_HD.tif

Un autre possibilité : l'option -tr permet d'imposer une résolution spatiale.

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -tr 750 750 -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_tr.tif
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -tr 750 750 -te -1120000 -630000 1120000 630000 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_tr.jpg

In [ ]:
!gdalinfo $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_tr.tif

In [ ]:
! identify $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_HD.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_tr.jpg

In [ ]:
im6 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_tr.jpg', 'r')
display(im6)

Pour comparer les différentes images, faisons un zoom sur l'ouest des Pyrenées

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6 lon_0=7.3' -te -112000 -63000 112000 63000 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom.jpg
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6 lon_0=7.3' -tr 750 750 -te -112000 -63000 112000 63000 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom.jpg
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6  lon_0=7.3' -ts 100 56 -te -112000 -63000 112000 63000 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom.jpg
im7 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom.jpg', 'r')
display(im7)
im8 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom.jpg', 'r')
display(im8)
im9 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom.jpg', 'r')
display(im9)

Et si on agrandit toutes ces images :

In [ ]:
!convert -quiet -resize 675x380! $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom.jpg  $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_675x380.jpg >/dev/null 2>&1
!convert -quiet -resize 675x380! $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom.jpg  $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom_675x380.jpg >/dev/null 2>&1
!convert -quiet -resize 675x380! $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom_675x380.jpg >/dev/null 2>&1

In [ ]:
im10 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_675x380.jpg', 'r')
display(im10)
im11 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_tr_zoom_675x380.jpg', 'r')
display(im11)
im12 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_ts_zoom_675x380.jpg', 'r')
display(im12)

Lors de découpes, la méthode de rééchantillonnage est aussi très importante !
Option -r : exemple -r cubic , -r lanczos (lanczos est parfois plus nette, mais il est plus long, et peut créer des artefacts. -r cubic est le meilleur compromis).
Sans option : pas de rééchantillonage

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6 lon_0=7.3'  -ts 1350 760 -te -56000 -31500 56000 31500 $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_1350x760.jpg
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.6 lon_0=7.3'  -ts 1350 760 -te -56000 -31500 56000 31500 -r cubic $INPUT $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_1350x760-rcubic.jpg

In [ ]:
im13 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_1350x760.jpg', 'r')
display(im13)
im14 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_HD_zoom_1350x760-rcubic.jpg', 'r')
display(im14)

## Visualisation des bandes d'une image RGB

In [ ]:
!gdal_translate -b 1 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeR.jpg 
!gdal_translate -b 2 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeG.jpg
!gdal_translate -b 3 $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeB.jpg

In [ ]:
im15 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeR.jpg', 'r')
display(im15)
im16 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeR.jpg', 'r')
display(im16)
im17 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_700x700_bandeR.jpg', 'r')
display(im17)

Les bandes visibles composant le produit Geocolor RGB sont visualisées. </br>
📌 -> <a href="../DOCS/mtg_fci_synthese_produits.pdf" target="_blank"> + d'info sur les recettes des produits RGB</a></br>
📌 -> <a href="../DOCS/MTG_fiche_geocolor_RGB.pdf" target="_blank"> Fiche Geocolor RGB</a>

<div class="alert alert-info alert-success">
<h3> 5 - Ajouter un contour, un logo, annoter l'image </h3>
</div>

La commande gdal_rasterize permet d'ajouter un contour.  Les frontières dans l'exemple ci-dessous </br>
**Attention** , la commande gdal_rasterize écrase l'image qui est donnée en entrée.
Les options -b 1 -burn 200 -b 2 -burn 200 -b 3 -burn 200 permettent d'indiquer la couleur RGB (255,255,255 = blanc; 0,0,0, = noir; 255,0,0 =rouge ...)

In [ ]:
!gdal_rasterize -b 1 -burn 200 -b 2 -burn 200 -b 3 -burn 200 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp  $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif >/dev/null 2>&1
!convert -quiet -resize 1920x1080 $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1.tif $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour.jpg
im18 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour.jpg', 'r')
display(im18)

Ajouter un titre sur une image

In [ ]:
titre="Meteosat-12 - Geocolor RGB - 22/04/2026 à 12 h UTC"
#Ajout bandeau bleu semi-transparent sous le futur titre
!composite -quiet -geometry +0+0 OUTILS/bandeau_bleu_1920x1080.png $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour.jpg $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_bandeau.jpg
!convert $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_bandeau.jpg -font OUTILS/Police_Marianne/Marianne-Bold.otf -pointsize 25 -fill '#eeeeee' -annotate +24+24 "$titre" $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_date.jpg >/dev/null 2>&1
im19 = Image.open(output + '/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_date.jpg', 'r')
display(im19)

Ajout du logo Météo-France

In [ ]:
!composite -quiet -geometry +1520+1016 OUTILS/logo_MF-400.png $output/globeMtruecolorGTP.mtg.20260422.1200.24_ortho_france_46.5_1_contour_date.jpg $output/globeMtruecolor.mtg.20260422.1200_france.jpg
im20 = Image.open(output + '/globeMtruecolor.mtg.20260422.1200_france.jpg', 'r')
display(im20)

<div class="alert alert-info alert-success">
<h3> 6 - Faire une animation </h3>
</div>

Exemple de boucle bash

In [ ]:
%%bash
output="RESULTS"
mkdir -p $output/tmp
rm -f  $output/tmp/*
for hh in {00..23}
do
    echo "hh=${hh},"
    INPUT="/stockage/DATA/TIF/20260422/EOXF71MTGG22${hh}00.globeMtruecolorGTP.mtg.20260422.${hh}00.*.tif.LT"
    # Découpe et reprojection 
    gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -ts 1920 1080 -r cubic $INPUT $output/tmp/geocolor.mtg.20260422_${hh}00_france.tif >/dev/null 2>&1
    # Ajout contour
    gdal_rasterize -b 1 -burn 200 -b 2 -burn 200 -b 3 -burn 200 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp $output/tmp/geocolor.mtg.20260422_${hh}00_france.tif >/dev/null 2>&1
    # Ajout bandeau 
    composite -quiet -geometry +0+0 OUTILS/bandeau_bleu_1920x1080.png $output/tmp/geocolor.mtg.20260422_${hh}00_france.tif $output/tmp/geocolor.mtg.20260422_${hh}00_france_contour_bandeau.tif #>/dev/null 2>&1
    # Ajout titre
    titre="Meteosat-12 - Geocolor RGB - 22/04/2026 à ${hh} h UTC"
    convert $output/tmp/geocolor.mtg.20260422_${hh}00_france_contour_bandeau.tif -font OUTILS/Police_Marianne/Marianne-Bold.otf -pointsize 25 -fill '#eeeeee' -annotate +24+24 "$titre" $output/tmp/geocolor.mtg.20260422_${hh}00_france.jpg >/dev/null 2>&1
done
rm -f $output/tmp/*tif
# Création de l'animation
ffmpeg -y -framerate 3 -pattern_type glob -i "$output/tmp/geocolor.mtg.20260422_*00_france.jpg"    -pix_fmt yuv420p -vcodec libx264 -crf 27 $output/geocolor_mtg_20260422_france.mp4 >/dev/null 2>&1

Quelques exemples d'animations réalisées : Galerie CMS</br>
⭐ Dossiers médiatiques :  http://e-sat.cms.meteo.fr/GALERIE/archive.html </br>
⭐ Animations temps-réel : http://e-sat.cms.meteo.fr/GALERIE/animations.html